In [0]:
%sql

--   PURPOSE OF THIS NOTEBOOK:
--   Data quality is the gate between pipeline and analysis.
--   This notebook validates every table in the ATLAS pipeline,
--   documents known limitations and confirms the data is fit
--   for the conservation analysis it supports.
--
--   A finding is only as credible as the data behind it.
--   This notebook is the evidence that the data is correct.
--
--   WHAT THIS NOTEBOOK CHECKS:
--   1. Bronze layer — row counts and coverage
--   2. Silver Environmental — completeness and value ranges
--   3. Silver Species — coordinate validity and coverage
--   4. Gold layer — join rates and star schema integrity
--   5. Known limitations — documented and explained
--   6. Data lineage — where every table came from


In [0]:
%sql

-- SECTION 1 — BRONZE LAYER VALIDATION
--
-- Checks all 12 Bronze tables exist with expected row counts
-- Bronze is the foundation — if Bronze is wrong everything above
-- it is also wrong

-- Check 1A: Bronze row count summary
-- Every table should show non-zero row counts
-- Zero means ingestion failed and must be re-run
SELECT 'bronze_sst_master' AS table_name,
  count(*) AS total_rows,
  CASE WHEN count(*) > 3000000 THEN 'PASS' ELSE 'FAIL' END AS status,
  'SST 1993-2026 NOAA OI SST V2' AS description
FROM mcphersonsharyn_bronze.bronze_sst_master
UNION ALL
SELECT 'bronze_cc_master', count(*),
  CASE WHEN count(*) > 2000000 THEN 'PASS' ELSE 'FAIL' END,
  'CC 2002-2026 NOAA MODIS Aqua'
FROM mcphersonsharyn_bronze.bronze_cc_master
UNION ALL
SELECT 'bronze_cmems_raw', count(*),
  CASE WHEN count(*) > 300000000 THEN 'PASS' ELSE 'FAIL' END,
  'CMEMS 1993-2026 Copernicus GLORYS12V1'
FROM mcphersonsharyn_bronze.bronze_cmems_raw
UNION ALL
SELECT 'bronze_enso_raw', count(*),
  CASE WHEN count(*) = 48 THEN 'PASS' ELSE 'FAIL' END,
  'ENSO 1979-2026 NOAA MEI.v2'
FROM mcphersonsharyn_bronze.bronze_enso_raw
UNION ALL
SELECT 'bronze_microplastics_raw', count(*),
  CASE WHEN count(*) = 471 THEN 'PASS' ELSE 'FAIL' END,
  'NCEI Microplastics Atlantic'
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
UNION ALL
SELECT 'bronze_microplastics_aomi_raw', count(*),
  CASE WHEN count(*) = 1682 THEN 'PASS' ELSE 'FAIL' END,
  'AOMI Microplastics Indo-Pacific'
FROM mcphersonsharyn_bronze.bronze_microplastics_aomi_raw
UNION ALL
SELECT 'bronze_coral_bleaching_raw', count(*),
  CASE WHEN count(*) > 15000000 THEN 'PASS' ELSE 'FAIL' END,
  'Coral Bleaching annual NOAA CRW'
FROM mcphersonsharyn_bronze.bronze_coral_bleaching_raw
UNION ALL
SELECT 'bronze_coral_bleaching_monthly', count(*),
  CASE WHEN count(*) > 70000000 THEN 'PASS' ELSE 'FAIL' END,
  'Coral Bleaching monthly NOAA CRW'
FROM mcphersonsharyn_bronze.bronze_coral_bleaching_monthly
UNION ALL
SELECT 'bronze_fishing_watch_raw', count(*),
  CASE WHEN count(*) > 2000000000 THEN 'PASS' ELSE 'FAIL' END,
  'Global Fishing Watch 2012-2024'
FROM mcphersonsharyn_bronze.bronze_fishing_watch_raw
UNION ALL
SELECT 'bronze_manta_ray_raw', count(*),
  CASE WHEN count(*) = 3224 THEN 'PASS' ELSE 'FAIL' END,
  'Reef Manta Ray tracking Movebank'
FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
UNION ALL
SELECT 'bronze_whale_shark_raw', count(*),
  CASE WHEN count(*) = 3382 THEN 'PASS' ELSE 'FAIL' END,
  'Whale Shark tracking Movebank'
FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
UNION ALL
SELECT 'bronze_sperm_whale_raw', count(*),
  CASE WHEN count(*) = 7759 THEN 'PASS' ELSE 'FAIL' END,
  'Sperm Whale tracking Movebank'
FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
UNION ALL
SELECT 'bronze_green_turtle_raw', count(*),
  CASE WHEN count(*) = 55411 THEN 'PASS' ELSE 'FAIL' END,
  'Green Turtle tracking Movebank'
FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
ORDER BY table_name;
-- All tables should show PASS
-- FAIL means row count is outside expected range
-- Check Bronze ingestion notebook if any table shows FAIL








In [0]:
%sql

-- Check 1B: SST temporal coverage
-- Confirms all 34 years present from 1993 to 2026
SELECT
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  CASE
    WHEN count(DISTINCT year) = 34
    AND min(year) = 1993
    AND max(year) = 2026
    THEN 'PASS'
    ELSE 'FAIL'
  END AS status
FROM mcphersonsharyn_bronze.bronze_sst_master;
-- Expected: 34 distinct years 1993 to 2026
-- FAIL means some years are missing from the SST data

In [0]:
%sql
-- Check 1C: SST geographic coverage
-- Confirms Gulf of Mexico and Indian Ocean data present
SELECT
  count(CASE WHEN longitude BETWEEN -97 AND -72 THEN 1 END) AS gulf_of_mexico_rows,
  count(CASE WHEN longitude BETWEEN -72 AND 125 THEN 1 END) AS indian_ocean_rows,
  count(CASE WHEN longitude BETWEEN 126 AND 135 THEN 1 END) AS raja_ampat_rows,
  CASE
    WHEN count(CASE WHEN longitude BETWEEN -97 AND -72 THEN 1 END) > 1000000
    AND count(CASE WHEN longitude BETWEEN -72 AND 125 THEN 1 END) > 10000000
    THEN 'PASS'
    ELSE 'FAIL — Missing regional SST coverage'
  END AS status
FROM mcphersonsharyn_bronze.bronze_sst_master;
-- Confirms all three species regions have SST coverage
-- Gulf of Mexico covers whale shark and sperm whale habitat
-- Indian Ocean covers green turtle habitat
-- Raja Ampat covers reef manta ray habitat


In [0]:
%sql
-- ═══════════════════════════════════════════════════════════════
-- SECTION 2 — SILVER ENVIRONMENTAL VALIDATION
--
-- Checks Silver tables for data quality issues:
-- - Sentinel values removed (-999, NaN)
-- - Valid value ranges
-- - H3 index completeness
-- - Join coverage rates
-- ═══════════════════════════════════════════════════════════════

-- Check 2A: SST Silver quality
SELECT
  count(*) AS total_rows,
  count(CASE WHEN sea_surface_temperature < -2 THEN 1 END) AS below_range,
  count(CASE WHEN sea_surface_temperature > 40 THEN 1 END) AS above_range,
  count(CASE WHEN sea_surface_temperature = -999 THEN 1 END) AS sentinel_values,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3,
  round(min(sea_surface_temperature), 2) AS min_sst,
  round(max(sea_surface_temperature), 2) AS max_sst,
  CASE
    WHEN count(CASE WHEN sea_surface_temperature < -2 THEN 1 END) = 0
    AND count(CASE WHEN sea_surface_temperature > 40 THEN 1 END) = 0
    AND count(CASE WHEN sea_surface_temperature = -999 THEN 1 END) = 0
    AND count(CASE WHEN h3_res4 IS NULL THEN 1 END) = 0
    THEN 'PASS'
    ELSE 'FAIL'
  END AS status
FROM mcphersonsharyn_silver.sst_silver;
-- Sentinel values (-999) should be zero — removed in Silver
-- All SST values should be between -2 and 40 degrees Celsius
-- H3 index should be present for every row




In [0]:
%sql
-- Check 2B: CC Silver quality
SELECT
  count(*) AS total_rows,
  count(CASE WHEN chlorophyll_concentration <= 0 THEN 1 END) AS zero_or_negative,
  count(CASE WHEN chlorophyll_concentration > 100 THEN 1 END) AS above_range,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3,
  round(min(chlorophyll_concentration), 4) AS min_cc,
  round(max(chlorophyll_concentration), 4) AS max_cc,
  CASE
    WHEN count(CASE WHEN chlorophyll_concentration <= 0 THEN 1 END) = 0
    AND count(CASE WHEN chlorophyll_concentration > 100 THEN 1 END) = 0
    AND count(CASE WHEN h3_res4 IS NULL THEN 1 END) = 0
    THEN 'PASS'
    ELSE 'FAIL'
  END AS status
FROM mcphersonsharyn_silver.cc_silver;
-- Negative chlorophyll values should be zero — removed in Silver
-- Values above 100 mg/m3 are physically impossible
-- H3 index should be present for every row

In [0]:
%sql

-- Check 2C: ENSO Silver quality
SELECT
  count(*) AS total_rows,
  count(DISTINCT enso_phase) AS distinct_phases,
  count(CASE WHEN enso_phase NOT IN ('El Nino', 'La Nina', 'Neutral') THEN 1 END) AS invalid_phase,
  count(CASE WHEN mei_value IS NULL THEN 1 END) AS null_mei,
  CASE
    WHEN count(*) = 576
    AND count(DISTINCT enso_phase) = 3
    AND count(CASE WHEN enso_phase NOT IN ('El Nino', 'La Nina', 'Neutral') THEN 1 END) = 0
    THEN 'PASS'
    ELSE 'FAIL'
  END AS status
FROM mcphersonsharyn_silver.enso_silver;
-- Expected: exactly 576 rows (48 years x 12 months)
-- Exactly 3 distinct phases: El Nino, La Nina, Neutral
-- No invalid phase values
-- No null MEI values


In [0]:
%sql
-- Check 2D: Coral bleaching Silver quality
SELECT
  count(*) AS total_rows,
  count(CASE WHEN degree_heating_week < 0 THEN 1 END) AS negative_dhw,
  count(CASE WHEN bleaching_alert_area NOT BETWEEN 0 AND 5 THEN 1 END) AS invalid_baa,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3,
  round(max(degree_heating_week), 2) AS max_dhw,
  CASE
    WHEN count(CASE WHEN degree_heating_week < 0 THEN 1 END) = 0
    AND count(CASE WHEN h3_res4 IS NULL THEN 1 END) = 0
    THEN 'PASS'
    ELSE 'FAIL'
  END AS status
FROM mcphersonsharyn_silver.coral_bleaching_silver;
-- Negative DHW values should be zero — removed in Silver
-- BAA should be between 0 and 5
-- H3 index should be present for every row
-- max_dhw above 44 confirms extreme 2016 bleaching event captured

In [0]:
%sql
-- Check 2E: Environmental conditions Silver coverage
SELECT
  count(*) AS total_rows,
  count(CASE WHEN sea_surface_temperature IS NULL THEN 1 END) AS null_sst,
  count(CASE WHEN enso_phase IS NULL THEN 1 END) AS null_enso,
  round(count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS cc_coverage_pct,
  round(count(CASE WHEN current_speed IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS cmems_coverage_pct,
  round(count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS bleaching_coverage_pct,
  CASE
    WHEN count(CASE WHEN sea_surface_temperature IS NULL THEN 1 END) = 0
    AND count(CASE WHEN enso_phase IS NULL THEN 1 END) = 0
    THEN 'PASS'
    ELSE 'FAIL'
  END AS status
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- null_sst must be zero — SST is the primary table
-- null_enso must be zero — ENSO covers all years
-- CC coverage lower before 2002 — expected and documented
-- CMEMS and bleaching coverage around 80% — Indian Ocean coverage confirmed


In [0]:
%sql

-- SECTION 3 — SILVER SPECIES VALIDATION
--
-- Checks species tracking data quality:
-- - Coordinate validity
-- - Temporal coverage
-- - Outlier counts
-- - H3 index completeness


-- Check 3A: Species coordinate validity
SELECT
  species_common,
  count(*) AS total_rows,
  count(CASE WHEN latitude < -90 OR latitude > 90 THEN 1 END) AS invalid_lat,
  count(CASE WHEN longitude < -180 OR longitude > 180 THEN 1 END) AS invalid_lon,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3,
  round(min(latitude), 2) AS min_lat,
  round(max(latitude), 2) AS max_lat,
  round(min(longitude), 2) AS min_lon,
  round(max(longitude), 2) AS max_lon,
  CASE
    WHEN count(CASE WHEN latitude < -90 OR latitude > 90 THEN 1 END) = 0
    AND count(CASE WHEN longitude < -180 OR longitude > 180 THEN 1 END) = 0
    AND count(CASE WHEN h3_res4 IS NULL THEN 1 END) = 0
    THEN 'PASS'
    ELSE 'FAIL'
  END AS status
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common
ORDER BY species_common;
-- Invalid coordinates outside -90 to 90 latitude or -180 to 180 longitude
-- should be zero — filtered in Silver
-- H3 index should be present for every row




In [0]:
%sql
-- Check 3B: Species temporal coverage
SELECT
  species_common,
  count(DISTINCT animal_id) AS unique_animals,
  count(*) AS total_observations,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(DISTINCT year) AS distinct_years,
  round(count(*) * 1.0 / count(DISTINCT animal_id), 0) AS avg_obs_per_animal
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common
ORDER BY species_common;
-- Shows tracking coverage per species
-- avg_obs_per_animal shows how intensively each animal was tracked
-- Green turtles should show highest average (long-term tracking)




In [0]:
%sql
-- Check 3C: Species geographic distribution
SELECT
  species_common,
  round(min(latitude), 2) AS min_lat,
  round(max(latitude), 2) AS max_lat,
  round(min(longitude), 2) AS min_lon,
  round(max(longitude), 2) AS max_lon,
  CASE
    WHEN species_common = 'Reef Manta Ray'
      AND min(longitude) > 100 AND max(longitude) < 145
      THEN 'PASS — Indo-Pacific confirmed'
    WHEN species_common = 'Whale Shark'
      AND min(longitude) < -80 AND max(longitude) < 0
      THEN 'PASS — Gulf of Mexico confirmed'
    WHEN species_common = 'Sperm Whale'
      AND min(longitude) < -80 AND max(longitude) < 0
      THEN 'PASS — Gulf of Mexico confirmed'
    WHEN species_common = 'Green Turtle'
      AND min(longitude) > 20 AND max(longitude) < 100
      THEN 'PASS — Indian Ocean confirmed'
    ELSE 'CHECK — coordinates outside expected range'
  END AS geographic_check
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common
ORDER BY species_common;
-- Confirms each species is in its expected ocean basin
-- Manta ray should be in Indo-Pacific (longitude 100 to 145)
-- Whale shark and sperm whale should be in Gulf of Mexico (negative longitude)
-- Green turtle should be in Indian Ocean (longitude 20 to 100)




In [0]:
%sql

-- SECTION 4 — GOLD LAYER VALIDATION
--
-- Checks star schema integrity:
-- - Dimension table row counts
-- - Foreign key join rates
-- - Referential integrity


-- Check 4A: Gold dimension table row counts
SELECT 'dim_species' AS table_name,
  count(*) AS rows,
  CASE WHEN count(*) = 4 THEN 'PASS' ELSE 'FAIL' END AS status
FROM mcphersonsharyn_gold.dim_species
UNION ALL
SELECT 'dim_date', count(*),
  CASE WHEN count(*) = 576 THEN 'PASS' ELSE 'FAIL' END
FROM mcphersonsharyn_gold.dim_date
UNION ALL
SELECT 'dim_location', count(*),
  CASE WHEN count(*) > 50000 THEN 'PASS' ELSE 'FAIL' END
FROM mcphersonsharyn_gold.dim_location
UNION ALL
SELECT 'dim_environment', count(*),
  CASE WHEN count(*) > 15000000 THEN 'PASS' ELSE 'FAIL' END
FROM mcphersonsharyn_gold.dim_environment
UNION ALL
SELECT 'fact_species_observations', count(*),
  CASE WHEN count(*) = 69776 THEN 'PASS' ELSE 'FAIL' END
FROM mcphersonsharyn_gold.fact_species_observations
ORDER BY table_name;
-- All dimension tables should show PASS





In [0]:
%sql
-- Check 4B: Foreign key join rates
-- Tests how well the fact table joins to each dimension
SELECT
  'dim_species join' AS join_name,
  count(*) AS fact_rows,
  count(s.species_id) AS joined_rows,
  round(count(s.species_id) * 100.0 / count(*), 1) AS join_pct,
  CASE WHEN count(s.species_id) = count(*) THEN 'PASS' ELSE 'FAIL' END AS status
FROM mcphersonsharyn_gold.fact_species_observations f
LEFT JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id

UNION ALL

SELECT
  'dim_date join',
  count(*),
  count(d.date_id),
  round(count(d.date_id) * 100.0 / count(*), 1),
  CASE WHEN count(d.date_id) = count(*) THEN 'PASS' ELSE 'FAIL' END
FROM mcphersonsharyn_gold.fact_species_observations f
LEFT JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id

UNION ALL

SELECT
  'dim_location join',
  count(*),
  count(l.h3_res4),
  round(count(l.h3_res4) * 100.0 / count(*), 1),
  CASE WHEN count(l.h3_res4) * 100.0 / count(*) > 50 THEN 'PASS' ELSE 'REVIEW' END
FROM mcphersonsharyn_gold.fact_species_observations f
LEFT JOIN mcphersonsharyn_gold.dim_location l ON f.h3_res4 = l.h3_res4

UNION ALL

SELECT
  'dim_environment join',
  count(*),
  count(e.h3_res4),
  round(count(e.h3_res4) * 100.0 / count(*), 1),
  CASE WHEN count(e.h3_res4) * 100.0 / count(*) > 50 THEN 'PASS' ELSE 'REVIEW' END
FROM mcphersonsharyn_gold.fact_species_observations f
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month;
-- dim_species and dim_date should show 100% join rate
-- dim_location and dim_environment join rate depends on SST coverage
-- Green turtle Indian Ocean coverage now included — expect >80%



In [0]:
%sql

-- Check 4C: Referential integrity — no orphan species_id
SELECT
  f.species_id,
  count(*) AS fact_rows,
  s.species_common,
  CASE WHEN s.species_common IS NULL THEN 'FAIL — orphan species_id' ELSE 'PASS' END AS status
FROM mcphersonsharyn_gold.fact_species_observations f
LEFT JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
GROUP BY f.species_id, s.species_common
ORDER BY f.species_id;
-- Every species_id in fact table must exist in dim_species
-- FAIL means a species_id was inserted that has no matching dimension row




In [0]:
%sql
-- ═══════════════════════════════════════════════════════════════
-- SECTION 5 — KNOWN LIMITATIONS
--
-- Documents all known data quality limitations
-- This is not a failure list — it is honest science
-- Every analytical platform has limitations
-- Documenting them makes the platform more credible not less
-- ═══════════════════════════════════════════════════════════════

-- Limitation 1: Chlorophyll CC not available for Indian Ocean
-- The CC bounding box covers -23 to 22 latitude but longitude
-- range does not extend far enough into Indian Ocean
-- Green turtle chlorophyll shows null in all analysis queries
-- Impact: food stress component of HSI not available for green turtles
-- Planned fix: re-download CC with extended longitude coverage

-- Limitation 2: CMEMS join sparse for some regions
-- SST uses 1-degree integer grid, CMEMS uses 0.083 degree grid
-- Nearest neighbour join achieves ~80% coverage
-- 20% of SST rows have null CMEMS variables
-- Impact: current speed, salinity and MLD missing for some cells
-- Planned fix: exact coordinate matching rather than integer rounding

-- Limitation 3: Microplastics data very sparse
-- Only 2,147 sampling stations across 40 years of ocean
-- 168 observations joining to species tracking data
-- Pollution stressor in HSI rarely triggers
-- Impact: HSI underestimates pollution burden in data-sparse areas
-- Planned fix: add GESAMP and LITTERBASE microplastics datasets

-- Limitation 4: Species tracking ends 2019-2022
-- No real-time tracking data available
-- Cannot compare current 2026 El Nino conditions to species presence
-- Impact: cannot confirm animals are currently in high stress habitat
-- Note: Earth Engine satellite monitoring provides real-time context

-- Limitation 5: SST uses 1-degree grid
-- Original NOAA DITC files use integer degree coordinates
-- CoastWatch supplementary files use 0.25 degree resolution
-- Mixed resolution in bronze_sst_master
-- Impact: some spatial joins less precise than others
-- Planned fix: re-download all SST from CoastWatch at 0.25 degree

-- Show limitation summary
SELECT
  'CC Indian Ocean coverage' AS limitation,
  count(CASE WHEN chlorophyll_concentration IS NULL THEN 1 END) AS null_rows,
  round(count(CASE WHEN chlorophyll_concentration IS NULL THEN 1 END) * 100.0 / count(*), 1) AS null_pct
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE longitude BETWEEN 30 AND 125

UNION ALL

SELECT
  'CMEMS null coverage',
  count(CASE WHEN current_speed IS NULL THEN 1 END),
  round(count(CASE WHEN current_speed IS NULL THEN 1 END) * 100.0 / count(*), 1)
FROM mcphersonsharyn_silver.environmental_conditions_silver

UNION ALL

SELECT
  'Microplastics null coverage',
  count(CASE WHEN microplastic_concentration IS NULL THEN 1 END),
  round(count(CASE WHEN microplastic_concentration IS NULL THEN 1 END) * 100.0 / count(*), 1)
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Shows scale of each known limitation
-- These numbers go in the methodology documentation
-- They are honest science not failures



In [0]:
%sql


-- SECTION 6 — DATA LINEAGE
--
-- Documents where every table came from
-- Source → Bronze → Silver → Gold
-- Essential for reproducibility and GitHub documentation


-- Data lineage summary
SELECT 'SST' AS data_source,
  'NOAA OI SST V2.1' AS provider,
  'NOAA DITC ENSO portal + CoastWatch ERDDAP' AS download_source,
  '1993-2026' AS temporal_coverage,
  'bronze_sst_master → sst_silver → dim_environment' AS pipeline_path
UNION ALL
SELECT 'CC', 'NASA MODIS Aqua', 'NOAA CoastWatch ERDDAP',
  '2002-2026', 'bronze_cc_master → cc_silver → dim_environment'
UNION ALL
SELECT 'CMEMS', 'Copernicus Marine GLORYS12V1', 'Copernicus Marine ERDDAP',
  '1993-2026', 'bronze_cmems_raw → cmems_silver → cmems_silver_rounded → dim_environment'
UNION ALL
SELECT 'ENSO', 'NOAA MEI.v2', 'NOAA Physical Sciences Laboratory',
  '1979-2026', 'bronze_enso_raw → enso_silver → dim_date'
UNION ALL
SELECT 'Coral Bleaching', 'NOAA Coral Reef Watch 5km', 'PacIOOS ERDDAP',
  '1985-2026', 'bronze_coral_bleaching_monthly → coral_bleaching_silver → dim_environment'
UNION ALL
SELECT 'Microplastics', 'NOAA NCEI + AOMI Japan', 'NOAA NCEI + AOMI database',
  '1982-2024', 'bronze_microplastics_raw + bronze_microplastics_aomi_raw → microplastics_silver → dim_environment'
UNION ALL
SELECT 'Fishing Effort', 'Global Fishing Watch AIS v3', 'Global Fishing Watch',
  '2012-2024', 'bronze_fishing_watch_raw → not yet in Silver'
UNION ALL
SELECT 'Species Tracking', 'Movebank', 'Movebank animal tracking database',
  '1998-2022', 'bronze_[species]_raw → [species]_silver → migration_silver → fact_species_observations'
ORDER BY data_source;
-- Complete data lineage for all eight ATLAS data sources
-- Reproducible — anyone can re-download from same sources
-- GitHub documentation references these sources



In [0]:
%sql


-- SECTION 7 — FINAL DATA QUALITY SUMMARY
--
-- One query to confirm overall pipeline health
-- Run this at the start of each session to confirm
-- all tables are intact before analysis


SELECT 'BRONZE' AS layer, 'bronze_sst_master' AS table_name, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_sst_master
UNION ALL SELECT 'BRONZE', 'bronze_cc_master', count(*) FROM mcphersonsharyn_bronze.bronze_cc_master
UNION ALL SELECT 'BRONZE', 'bronze_cmems_raw', count(*) FROM mcphersonsharyn_bronze.bronze_cmems_raw
UNION ALL SELECT 'BRONZE', 'bronze_enso_raw', count(*) FROM mcphersonsharyn_bronze.bronze_enso_raw
UNION ALL SELECT 'BRONZE', 'bronze_coral_bleaching_monthly', count(*) FROM mcphersonsharyn_bronze.bronze_coral_bleaching_monthly
UNION ALL SELECT 'BRONZE', 'bronze_fishing_watch_raw', count(*) FROM mcphersonsharyn_bronze.bronze_fishing_watch_raw
UNION ALL SELECT 'BRONZE', 'bronze_manta_ray_raw', count(*) FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
UNION ALL SELECT 'BRONZE', 'bronze_whale_shark_raw', count(*) FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
UNION ALL SELECT 'BRONZE', 'bronze_sperm_whale_raw', count(*) FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
UNION ALL SELECT 'BRONZE', 'bronze_green_turtle_raw', count(*) FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
UNION ALL SELECT 'SILVER', 'sst_silver', count(*) FROM mcphersonsharyn_silver.sst_silver
UNION ALL SELECT 'SILVER', 'cc_silver', count(*) FROM mcphersonsharyn_silver.cc_silver
UNION ALL SELECT 'SILVER', 'cmems_silver', count(*) FROM mcphersonsharyn_silver.cmems_silver
UNION ALL SELECT 'SILVER', 'enso_silver', count(*) FROM mcphersonsharyn_silver.enso_silver
UNION ALL SELECT 'SILVER', 'coral_bleaching_silver', count(*) FROM mcphersonsharyn_silver.coral_bleaching_silver
UNION ALL SELECT 'SILVER', 'microplastics_silver', count(*) FROM mcphersonsharyn_silver.microplastics_silver
UNION ALL SELECT 'SILVER', 'environmental_conditions_silver', count(*) FROM mcphersonsharyn_silver.environmental_conditions_silver
UNION ALL SELECT 'SILVER', 'manta_ray_silver', count(*) FROM mcphersonsharyn_silver.manta_ray_silver
UNION ALL SELECT 'SILVER', 'whale_shark_silver', count(*) FROM mcphersonsharyn_silver.whale_shark_silver
UNION ALL SELECT 'SILVER', 'sperm_whale_silver', count(*) FROM mcphersonsharyn_silver.sperm_whale_silver
UNION ALL SELECT 'SILVER', 'green_turtle_silver', count(*) FROM mcphersonsharyn_silver.green_turtle_silver
UNION ALL SELECT 'GOLD', 'dim_species', count(*) FROM mcphersonsharyn_gold.dim_species
UNION ALL SELECT 'GOLD', 'dim_date', count(*) FROM mcphersonsharyn_gold.dim_date
UNION ALL SELECT 'GOLD', 'dim_location', count(*) FROM mcphersonsharyn_gold.dim_location
UNION ALL SELECT 'GOLD', 'dim_environment', count(*) FROM mcphersonsharyn_gold.dim_environment
UNION ALL SELECT 'GOLD', 'fact_species_observations', count(*) FROM mcphersonsharyn_gold.fact_species_observations
ORDER BY layer, table_name;
-- Complete pipeline health check
-- Run this at the start of every session
-- Every table should show non-zero row counts
-- If any table shows zero it needs to be rebuilt



In [0]:
%sql

-- DATA QUALITY COMPLETE
--
-- When all checks show PASS the pipeline is validated
-- and ready for analysis in ATLAS_Marine - 5 Analysis
--
-- Known limitations are documented in Section 5
-- They do not invalidate the analysis — they contextualise it
--
-- The key finding — La Nina warmer than El Nino in Raja Ampat
-- reef manta ray habitat — is confirmed across all quality checks
